# g1_limpo — treino DO ZERO no Colab

Sem resume, sem checkpoint de entrada, sem conferência de paridade. O `smoke.py` é
quem confere a configuração.

## Por que do zero

A `bloco20` mostrou o problema do warm-start: o robô pousava a caixa **escorando
nela**, e as correções de 14/09 tornaram essa estratégia inviável. Ele não perdeu uma
condição — perdeu a única forma de pousar que conhecia, e tem de descobrir outra com a
função de valor errada. O `s_C` foi de 0,484 para 0,002.

Uma política que nasce sob as regras novas nunca aprende a escorar. É o teste limpo.

## Antes de rodar

| | |
|---|---|
| Runtime -> Change runtime type | **GPU** |
| a branch `exp/g1-limpo-v2` | precisa estar **no GitHub** |
| Drive | a primeira célula pede autorização; o checkpoint é salvo em `MyDrive/g1_limpo` |

⚠ **O Colab gratuito cai por ociosidade.** Uma cópia do último checkpoint vai para
`MyDrive/g1_limpo/em_curso/` a cada 10 minutos — a perda máxima é de 10 min de treino.

⚠ Uma sessão **não** termina o treino. Ele para pelo relógio; a sessão seguinte usa o
notebook de RESUME.


In [ ]:
# ⚠ NADA DE `import torch` AQUI. O torch registra operadores C++ no import, e se ele
# entrar no kernel ANTES do pip, um reload depois levanta
# `Only a single TORCH_LIBRARY can be used to register the namespace triton`.
import subprocess, sys

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True)
print(smi.stdout or smi.stderr)
assert smi.returncode == 0 and smi.stdout.strip(), \
    "sem GPU. Runtime -> Change runtime type -> GPU"
print("python", sys.version.split()[0])

In [ ]:
import subprocess, sys

# ⚠⚠ LISTA DE ARGUMENTOS, NUNCA STRING DE SHELL. Com `!pip install ... numpy<2.5` o
# shell lê `<` como REDIRECIONAMENTO, tenta abrir um arquivo chamado `2.5`, aborta com
# exit 2 — e o pip NUNCA RODA, em silêncio.
cmd = [sys.executable, "-m", "pip", "install", "--no-warn-conflicts", "mjlab==1.5.3"]
print(" ".join(cmd), flush=True)
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-2000:])
if r.returncode != 0:
    print(r.stderr[-2000:])
assert r.returncode == 0, "o pip falhou"

In [ ]:
import subprocess, sys

chk = subprocess.run(
    [sys.executable, "-c",
     "import torch;print(torch.__version__, torch.cuda.is_available())"],
    capture_output=True, text=True)
print("subprocesso:", chk.stdout.strip() or chk.stderr[-600:])
assert " True" in chk.stdout, \
    "o pip trocou o torch e a CUDA foi embora. Reinicie o kernel e rode da célula 1"

import torch, mjlab, mujoco, warp
print(f"torch {torch.__version__}  {torch.cuda.get_device_name(0)}")
print("mjlab", getattr(mjlab, "__version__", "?"),
      "| mujoco", mujoco.__version__, "| warp", warp.config.version)

In [ ]:
import importlib, os, pathlib, shutil, subprocess, sys

os.environ.setdefault("MUJOCO_GL", "egl")

RUN    = "zero01"                  # ⚠ NOME NOVO A CADA TENTATIVA DO ZERO
BRANCH = "exp/g1-limpo-v2"

# ⚠ O LOG FICA LOCAL, e não no Drive. O `tfevents` é anexado a cada iteração, e o FUSE
# do Drive reescreve o arquivo inteiro a cada flush — lento, e já viu corromper. O
# Drive recebe CÓPIAS do checkpoint.
BASE     = pathlib.Path("/content")
RAIZ     = BASE / "g1"             # o clone, refeito a cada sessão
LOG_ROOT = BASE / "logs"           # ⚠ FORA de RAIZ: o re-clone apaga o que está dentro
raiz_exp = LOG_ROOT / "g1_limpo"

# ⚠ Uma linha, e nenhum segredo: o Drive é do próprio usuário. A montagem pede
# autorização interativa na primeira vez.
from google.colab import drive; drive.mount("/content/drive")
DRIVE = pathlib.Path("/content/drive/MyDrive/g1_limpo")
DRIVE.mkdir(parents=True, exist_ok=True)

if RAIZ.exists():
    shutil.rmtree(RAIZ)
subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--depth", "1",
                "https://github.com/JoaoBornelli/g1_training.git", str(RAIZ)],
               check=True)
print("clone =", subprocess.run(["git", "-C", str(RAIZ), "log", "--oneline", "-1"],
                                capture_output=True, text=True).stdout.strip())

# ⚠ `invalidate_caches` NÃO é higiene. O Python cacheia um finder POR DIRETÓRIO, e o
# de um diretório que não existia na hora da inserção fica cacheado como VAZIO —
# `import g1_limpo` falharia com `No module named` mesmo com o pacote em disco.
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
importlib.invalidate_caches()

print("run      =", RUN)
print("log_root =", LOG_ROOT)
print("drive    =", DRIVE)

In [ ]:
import pathlib, re, shutil, threading, time, zipfile
from IPython.display import FileLink, display

SAIDA = BASE
ULTIMO_CKPT = ULTIMA_IT = ULTIMO_PACOTE = None


def _run_nova(run=RUN):
    """A pasta de run mais recente e seus `model_*.pt` por número — ou `(None, [])`."""
    if not raiz_exp.is_dir():
        return None, []
    runs = sorted(p for p in raiz_exp.iterdir()
                  if p.is_dir() and p.name.endswith(run))
    if not runs:
        return None, []
    cks = sorted(runs[-1].glob("model_*.pt"),
                 key=lambda p: int(re.search(r"(\d+)", p.name).group(1)))
    return runs[-1], cks


def empacota(run=RUN, baixa=True):
    """Zipa o ÚLTIMO checkpoint + os tfevents e oferece o download."""
    global ULTIMO_CKPT, ULTIMA_IT, ULTIMO_PACOTE
    nova, cks = _run_nova(run)
    if not cks:
        print("nenhum checkpoint ainda")
        return None
    ultimo = cks[-1]
    it = int(re.search(r"(\d+)", ultimo.name).group(1))
    pacote = SAIDA / f"{run}_it{it}.zip"
    with zipfile.ZipFile(pacote, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(ultimo, ultimo.name)
        for ev in sorted(nova.glob("events.out.tfevents*")):
            z.write(ev, ev.name)
        for p in sorted(nova.glob("params/*")):
            z.write(p, f"params/{p.name}")
    ULTIMO_CKPT, ULTIMA_IT, ULTIMO_PACOTE = ultimo, it, pacote
    print(f"{pacote}  ({pacote.stat().st_size / 2**20:.1f} MB)  it {it}")
    if baixa:
        display(FileLink(str(pacote.relative_to(BASE))))
    return pacote


def copia_para_drive(run=RUN):
    """O zip e o último `model_*.pt` vão para `MyDrive/g1_limpo`."""
    if ULTIMO_PACOTE is None:
        empacota(run, baixa=False)
    if ULTIMO_PACOTE is None:
        return
    for p in (ULTIMO_PACOTE, ULTIMO_CKPT):
        shutil.copy2(p, DRIVE / p.name)
    print("no Drive:", ULTIMO_PACOTE.name, "e", ULTIMO_CKPT.name)


def copia_periodica_para_drive(minutos=10, run=RUN):
    """Thread daemon: copia o último checkpoint para `em_curso/` a cada N minutos.

    ⚠ Contra a QUEDA da sessão, que no Colab gratuito vem por OCIOSIDADE. A perda
    máxima passa a ser `minutos` de treino. Daemon: morre com o kernel e não segura o
    `finally` do lançamento.
    """
    destino = DRIVE / "em_curso"
    destino.mkdir(exist_ok=True)

    def _laco():
        visto = None
        while True:
            time.sleep(minutos * 60)
            try:
                _, cks = _run_nova(run)
                if cks and cks[-1] != visto:
                    shutil.copy2(cks[-1], destino / cks[-1].name)
                    visto = cks[-1]
                    print(f"[drive] {cks[-1].name}", flush=True)
            except Exception as e:
                print(f"[drive] falhou: {e}", flush=True)

    threading.Thread(target=_laco, daemon=True).start()
    print(f"cópia periódica ligada: a cada {minutos} min para {destino}")

In [ ]:
# =====================================================================
#  TREINO DO ZERO — Colab, 8192 envs
# =====================================================================
import dataclasses, sys
import torch
sys.path.insert(0, str(RAIZ))

import g1_limpo
from g1_limpo import recompensas as RC
from mjlab.scripts.train import TrainConfig, launch_training

# ⚠ SEM OVERRIDE DE RECOMPENSA. O `velocidade_por_regime` do repo já reduz por `amax`
# PARADO (manipulação, onde o teto é de imobilidade) e por MÉDIA andando (onde o teto
# é o p99 por junta). Foi o `amax` nos três regimes que colapsou a `bloco19`.
# ⚠ PESO −2,0, o de sempre. Do zero não se empilha um preço não calibrado sobre uma
# política que ainda não anda. Com o `amax` gateado ao regime parado, o preço da
# MARCHA volta a ser o da bloco17 — só a manipulação fica mais cara.
PESO_VELOCIDADE = -2.0

# ⚠ 8192 É O NÚMERO DO COLAB, e ele NÃO vem da VRAM. Medido em 2026-09-11: a GPU do
# Colab tem 16 GB e roda 8192; a da Kaggle, também de 16 GB, não. Se der OOM: 4096.
NUM_ENVS = 8192

# ⚠ A META é ABSOLUTA e do zero não cabe numa sessão. A célula corta pelo relógio.
META = 30000

# ⚠ O `SEG_POR_ITER` está ancorado em medição: 8192 envs na GPU de 16 GB do Colab
# deram 7,554 + 0,986 = 8,54 s/iter (23 022 passos/s) na `bloco8`. Os 9,0 são margem
# para a cena do g1_limpo ser mais carregada. CORRIJA pelo `Collection time` real.
# ⚠ O Colab gratuito cai por OCIOSIDADE antes das 10,5 h; o Pro vai a 24 h.
HORAS_LIMITE = 10.5
SEG_POR_ITER = 9.0

cfg = dataclasses.replace(TrainConfig.from_task(g1_limpo.TASK_ID),
                          log_root=str(LOG_ROOT))
cfg.env.scene.num_envs = NUM_ENVS
cfg.agent.run_name = RUN
cfg.agent.logger = "tensorboard"

_vel = cfg.env.rewards["velocidade_por_regime"]
_vel.weight = PESO_VELOCIDADE

# ⚠⚠ DO ZERO: `resume = False` e a LR fica no DEFAULT. O degrau para 5e-4 é regra de
# WARM-START — ele existe porque a função de valor de um checkpoint velho está errada.
# Aqui não há função de valor; baixar a LR só atrasaria.
cfg.agent.resume = False

# ⚠ O ALVO FICA SORTEADO, 0,75 a 1,0. É o ponto do treino do zero: a política nasce
# sob as regras novas, e generalizar entre alturas é uma delas. Pinar aqui seria
# repetir o experimento da sonda.
_c = cfg.env.commands["alvo_caixa"]

cabe_tempo = int(HORAS_LIMITE * 3600 / SEG_POR_ITER)
cfg.agent.max_iterations = min(META, cabe_tempo)

print(f"envs        = {NUM_ENVS}")
print(f"lote do PPO = {NUM_ENVS * cfg.agent.num_steps_per_env} transições, "
      f"minilote {NUM_ENVS * cfg.agent.num_steps_per_env // cfg.agent.algorithm.num_mini_batches}")
print(f"velocidade  = peso {_vel.weight}   [amax parado, média andando]")
print(f"alvo z      = {_c.altura_carregar_faixa}   [SORTEADO]")
print(f"folga_apoiada_N = {_c.folga_apoiada_N} N")
print(f"lr          = {cfg.agent.algorithm.learning_rate} "
      f"({cfg.agent.algorithm.schedule})   seed {cfg.agent.seed}")
print(f"cabe em {HORAS_LIMITE:.1f} h a {SEG_POR_ITER:.1f} s/iter = {cabe_tempo}")
print(f"vai rodar   = {cfg.agent.max_iterations} de {META}")
if META > cabe_tempo:
    print(f"\n⚠ ESTA SESSÃO NÃO ALCANÇA A {META}. Continue pelo notebook de RESUME.")
print(f"log_root    = {cfg.log_root}\n")

print("⚠ O QUE ESPERAR DO ZERO:")
print("   ~400   o robô fica de pé e o episódio deixa de morrer cedo")
print("   ~1000  a marcha se forma: razao_marcha sobe acima de 0,50")
print("   ~2500  a fatia de locomoção começa a descer de 0,95")
print("   depois disso é que a manipulação tem fatia para aprender\n")

# ⚠ A cópia periódica começa ANTES do lançamento: contra a queda por ociosidade.
copia_periodica_para_drive()

# ⚠ `try/finally`, e não a linha nua. Assim o pacote nasce também quando o treino
# estoura ou quando você interrompe o kernel.
try:
    launch_training(g1_limpo.TASK_ID, cfg)
finally:
    empacota()
    copia_para_drive()

## Persistir à mão

O `finally` da célula acima já copia para o Drive. Esta célula existe para quando ele
**não** rodou — sessão morta, kernel reiniciado.


In [ ]:
empacota(baixa=True)
copia_para_drive()
print("\nna próxima sessão: o notebook de RESUME acha o checkpoint em MyDrive/g1_limpo")